# Визуализация результатов модели ConvBiGRUDetector

Этот ноутбук предназначен для визуализации результатов работы модели ConvBiGRUDetector и сравнения их с оригинальной разметкой.

## Импорт необходимых библиотек

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

# Добавляем путь к src для импорта модулей
PROJECT_ROOT = Path.cwd().resolve().parent  # если ноутбук лежит в notebooks
sys.path.insert(0, str(PROJECT_ROOT))

# Импорты из проекта
from src.modeling.model_registry import get_model
from src.data_loading.epilepsy_datamodule import EpilepsyDataModule
from src.data_loading.seizure_annotation_reader import SeizureAnnotationReader

# Импорт функций постобработки
from experiments.postprocess_predictions import postprocess_predictions, predict_full_recording

# Параметры отображения
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 12

## Загрузка модели

In [ ]:
#ConvBiGRUDetector - last-v7
#UNet1DDetector - last-v11 
model = get_model('ConvBiGRUDetector', {})
print(f"Модель загружена: {type(model).__name__}")
print(f"Количество параметров: {sum(p.numel() for p in model.parameters()):,}")

ckpt_path = "../experiments/exp_001/checkpoints/last-v7.ckpt"
ckpt = torch.load(ckpt_path, map_location="cpu")

# если это Lightning-чекпойнт, веса обычно тут:
state_dict = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt

# убираем префикс "model."
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("model."):
        new_state_dict[k[len("model."):]] = v
    else:
        new_state_dict[k] = v  # на случай, если часть ключей без префикса
model.load_state_dict(new_state_dict)
model.eval()

## Загрузка данных

In [ ]:
# Инициализация датамодуля

train_animals=["Ati5x1", "Dex1x2NE", "Ati5y2"]  # Список ID животных для обучения
val_animals=["Ati5y1"]    # Список ID животных для валидации
test_animals=["Dex4x5"]   # Список ID животных для тестирования

data_module = EpilepsyDataModule(
    data_dir="../data/processed",
    window_length=2000,
    batch_size=64,
    overlap=0,
     train_animals=train_animals,
     val_animals=val_animals,
     test_animals=test_animals,
)

# Подготовка данных
data_module.prepare_data()
data_module.setup('test')

print(f"Размер тестовой выборки: {len(data_module.test_dataset)}")

## Визуализация результатов

In [ ]:
def visualize_prediction(signal, true_labels, predicted_probs, segments, title="Результаты предсказания"):
    """Визуализация сигнала, истинных меток и предсказанных сегментов"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    
    # Отображение сигнала (первый канал)
    time = np.arange(len(signal[0])) / 400  # время в секундах (частота дискретизации 400 Гц)
    ax1.plot(time, signal[0], 'b-', linewidth=0.8, label='Сигнал (канал 1)')
    ax1.set_ylabel('Амплитуда')
    ax1.set_title(f'{title} - Сигнал')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Отображение истинных меток
    ax2.plot(time, true_labels, 'g-', linewidth=1, label='Истинные метки')
    
    # Отображение предсказанных вероятностей
    ax2.plot(time, predicted_probs, 'r-', linewidth=1, label='Предсказанные вероятности')
    
    # Отображение сегментов
    for start, end in segments:
        start_time = start / 400
        end_time = end / 400
        ax2.axvspan(start_time, end_time, color='yellow', alpha=0.3, label='Предсказанные сегменты' if start == segments[0][0] else "")
    
    ax2.set_xlabel('Время (секунды)')
    ax2.set_ylabel('Вероятность / Метки')
    ax2.set_title(f'{title} - Предсказания и истинные метки')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Пример предсказания на тестовых данных

In [ ]:
# Получение случайного примера из тестовой выборки
test_sample = data_module.test_dataset[347]
signal, true_label = test_sample

# Добавляем размерность батча
signal_batch = signal.unsqueeze(0)

# Получение предсказания
model.eval()
with torch.no_grad():
    logits = model(signal_batch)
    predicted_probs = torch.sigmoid(logits).squeeze().numpy()

# Постобработка
segments = postprocess_predictions(predicted_probs)

# Визуализация
visualize_prediction(
    signal.numpy(), 
    true_label.numpy(), 
    predicted_probs, 
    segments, 
    "Пример предсказания ConvBiGRUDetector"
)

## Сравнение с оригинальной разметкой

In [ ]:
def compare_with_ground_truth(signal, true_labels, predicted_probs, segments, sr=400):
    """Сравнение предсказанных сегментов с истинной разметкой"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    
    time = np.arange(len(signal[0])) / sr
    
    # Отображение сигнала
    ax1.plot(time, signal[0], 'b-', linewidth=0.5)
    ax1.set_ylabel('Амплитуда')
    ax1.set_title('Сигнал')
    ax1.grid(True, alpha=0.3)
    
    # Отображение истинных меток
    ax2.plot(time, true_labels, 'g-', linewidth=2, label='Истинные метки')
    
    # Отображение предсказанных вероятностей
    ax2.plot(time, predicted_probs, 'r-', linewidth=1, alpha=0.7, label='Предсказанные вероятности')
    
    # Отображение истинных сегментов
    # Здесь предполагается, что истинные сегменты можно получить из true_labels
    # В реальной реализации это может потребовать дополнительной обработки
    true_segments = []
    in_seizure = False
    start = 0
    
    for i, label in enumerate(true_labels):
        if not in_seizure and label == 1:
            in_seizure = True
            start = i
        elif in_seizure and label == 0:
            in_seizure = False
            true_segments.append((start, i))
    
    if in_seizure:
        true_segments.append((start, len(true_labels)))
    
    # Отображение истинных сегментов
    for start, end in true_segments:
        start_time = start / sr
        end_time = end / sr
        ax2.axvspan(start_time, end_time, color='green', alpha=0.2, label='Истинные сегменты' if start == true_segments[0][0] else "")
    
    # Отображение предсказанных сегментов
    for start, end in segments:
        start_time = start / sr
        end_time = end / sr
        ax2.axvspan(start_time, end_time, color='red', alpha=0.3, label='Предсказанные сегменты' if start == segments[0][0] else "")
    
    ax2.set_xlabel('Время (секунды)')
    ax2.set_ylabel('Вероятность / Метки')
    ax2.set_title('Сравнение предсказанных и истинных сегментов')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return true_segments

In [ ]:
# Сравнение для того же примера
true_segments = compare_with_ground_truth(
    signal.numpy(), 
    true_label.numpy(), 
    predicted_probs, 
    segments
)

## Анализ производительности

In [ ]:
def calculate_metrics(true_segments, predicted_segments, signal_length, sr=400):
    """Расчет метрик точности для предсказанных сегментов"""
    # Создание бинарных масок для истинных и предсказанных сегментов
    true_mask = np.zeros(signal_length, dtype=bool)
    pred_mask = np.zeros(signal_length, dtype=bool)
    
    for start, end in true_segments:
        true_mask[start:end] = True
        
    for start, end in predicted_segments:
        pred_mask[start:end] = True
    
    # Расчет метрик
    tp = np.sum(true_mask & pred_mask)
    fp = np.sum(~true_mask & pred_mask)
    fn = np.sum(true_mask & ~pred_mask)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'true_events': len(true_segments),
        'predicted_events': len(predicted_segments)
    }

In [ ]:
# Расчет метрик для примера
metrics = calculate_metrics(true_segments, segments, len(signal[0]))

print("Метрики производительности:")
print(f"Точность (Precision): {metrics['precision']:.3f}")
print(f"Полнота (Recall): {metrics['recall']:.3f}")
print(f"F1-мера: {metrics['f1_score']:.3f}")
print(f"Истинных событий: {metrics['true_events']}")
print(f"Предсказанных событий: {metrics['predicted_events']}")

## Заключение

Этот ноутбук демонстрирует:
1. Загрузку модели ConvBiGRUDetector
2. Визуализацию результатов предсказания на тестовых данных
3. Сравнение предсказанных сегментов с оригинальной разметкой
4. Расчет метрик производительности

Для более полного анализа можно:
1. Протестировать модель на большем количестве примеров
2. Рассчитать средние метрики по всей тестовой выборке
3. Визуализировать ошибки модели (ложные срабатывания и пропуски)
4. Проанализировать производительность на разных типах записей